# Demo 2: Build a serverless generative-AI API

In this demo, an HTTP request starts a synchronous AWS workflow:

**Notebook client → Amazon API Gateway → AWS Lambda → Amazon Bedrock → HTTP response**

The notebook sends a prompt to a public HTTP endpoint. API Gateway converts the request into a Lambda event, Lambda validates the input and calls Amazon Nova Micro through Bedrock's Converse API, and the generated answer returns through the same path. CloudWatch Logs records what the Lambda function did.

Unlike Demo 1's asynchronous image pipeline, this request is synchronous: the notebook waits for the model response.

## Learning goals and architecture

By the end of this notebook, you should be able to explain how an HTTP request reaches a foundation model through a serverless API, how prompting differs from model training, and which parts of a generative-AI application remain our responsibility.

The services and components have distinct roles:

- **Amazon API Gateway** provides the HTTP endpoint and forwards `POST /generate` requests to Lambda.
- **AWS Lambda** validates the request, constructs the model call, and converts the result into an HTTP response.
- **Amazon Bedrock** provides managed API access to foundation models.
- **Amazon Nova Micro** is the text model selected by this application.
- **The Converse API** supplies a common message-based interface for supported Bedrock models.
- **Amazon CloudWatch Logs** records successful invocations and errors from Lambda.
- **AWS SAM and CloudFormation** define and deploy the API, function, permissions, and integration.

A **foundation model** is a large, general-purpose machine-learning model trained in advance on broad and diverse data. Instead of being built for only one narrow task, it can use natural-language instructions in a prompt to perform many downstream tasks, such as explaining, summarising, or classifying text.

This is the key conceptual difference from Demo 1: Rekognition exposes specialised, predefined computer-vision operations such as label detection, whereas Demo 2 sends different instructions to the same general-purpose text model. Changing a prompt changes the request and may change the response; it does not retrain or modify the foundation model's weights.

In [6]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

import boto3
from IPython.display import JSON, display

REGION = "ap-southeast-2"
STACK_NAME = "cits5503-demo2"
PROFILE = os.environ.get("AWS_PROFILE")  # Set this in your terminal if needed.

# This lets the notebook work whether Jupyter starts here or at the repository root.
def locate_demo_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "demo2_bedrock_api",
        Path.cwd() / "demo" / "demo2_bedrock_api",
    ]
    for candidate in candidates:
        if (candidate / "template.yaml").exists():
            return candidate.resolve()
    raise RuntimeError("Open this notebook from its folder or the repository root.")

DEMO_DIR = locate_demo_dir()
DEMO_ROOT = DEMO_DIR.parent
# Reuse one configured session so every AWS client uses the same account and region.
session = boto3.Session(profile_name=PROFILE, region_name=REGION)
DEMO_DIR

PosixPath('/Users/larryhh/Documents/UWA/CITS5503/demo/demo2_bedrock_api')

## 1. Deployment prerequisite

Before running the API cells, execute `./manage_demo.sh --demo 2 --action deploy --profile cits5503` from the repository's `demo/` directory (omit `--profile` if you use the default AWS profile).

The stack creates a small public HTTP API, a Lambda function, and a role that permits the function to invoke only the selected Nova Micro model. After changing the Lambda code, use the script's `redeploy` action to build and update the existing stack.

This endpoint intentionally has no authentication and allows requests from any web origin, so it is suitable only for this short-lived classroom demo. Do not use this configuration for a production API.

In [7]:
profile_option = f" --profile {PROFILE}" if PROFILE else ""
print("Required deployment command (run from a terminal):")
print(f"cd {DEMO_ROOT} && ./manage_demo.sh --demo 2 --action deploy{profile_option}")

Required deployment command (run from a terminal):
cd /Users/larryhh/Documents/UWA/CITS5503/demo && ./manage_demo.sh --demo 2 --action deploy


## 2. Preflight

Confirm which AWS account and region the SDK will use, then verify that the CloudFormation stack exists. This is a read-only check. The account must also be permitted to invoke the selected Nova Micro model through Amazon Bedrock.

In [8]:
identity = session.client("sts").get_caller_identity()
cloudformation = session.client("cloudformation")
cloudformation.describe_stacks(StackName=STACK_NAME)
print(f"Account: {identity['Account']}\nARN: {identity['Arn']}\nRegion: {REGION}\nAPI stack: {STACK_NAME}")

Account: 489389878001
ARN: arn:aws:iam::489389878001:user/larry.huynh@uwa.edu.au
Region: ap-southeast-2
API stack: cits5503-demo2


## 3. Read the generated API endpoint

CloudFormation generates the physical API and Lambda names. This cell reads those values from the deployed stack outputs. The notebook acts like an ordinary HTTP client: after obtaining the URL, it does not need AWS credentials to send a prompt to this intentionally public endpoint.

In [9]:
stack = cloudformation.describe_stacks(StackName=STACK_NAME)["Stacks"][0]
# Convert CloudFormation's list of key/value objects into an easier lookup dictionary.
outputs = {item["OutputKey"]: item["OutputValue"] for item in stack["Outputs"]}
API_URL = outputs["GenerateUrl"]
FUNCTION = outputs["FunctionName"]
display(JSON({"generate_url": API_URL, "function": FUNCTION}))

<IPython.core.display.JSON object>

## 4. How SAM connects the HTTP API to Lambda

AWS Serverless Application Model (SAM) is used at deployment time. The project's [`template.yaml`](template.yaml) declares an HTTP API and a Lambda function as infrastructure as code:

```yaml
BedrockApi:
  Type: AWS::Serverless::HttpApi

GenerateFunction:
  Type: AWS::Serverless::Function
  Properties:
    CodeUri: src/
    Handler: app.lambda_handler
    Events:
      Generate:
        Type: HttpApi
        Properties:
          ApiId: !Ref BedrockApi
          Path: /generate
          Method: POST
```

The `Events` block tells SAM to connect `POST /generate` to `app.lambda_handler` in `src/app.py`. SAM expands this shorthand into the API Gateway integration and the permission required for API Gateway to invoke Lambda.

The template passes the model ID and maximum prompt length into Lambda as environment variables. Its IAM policy permits `bedrock:InvokeModel` only for that selected model, rather than granting unrestricted Bedrock access.

### Runtime request flow

```mermaid
flowchart LR
    A[Notebook client] -->|POST /generate with JSON prompt| B[API Gateway]
    B -->|Lambda event| C[Lambda app.lambda_handler]
    C -->|Converse request| D[Amazon Bedrock / Nova Micro]
    D -->|Generated text and token usage| C
    C -->|JSON response| B
    B -->|HTTP response| A
    C -->|Completion or error log| E[CloudWatch Logs]
```

Lambda extracts the prompt from the HTTP body, rejects invalid input, and calls Bedrock's Converse API with a user message and an inference configuration. `maxTokens` limits the response length, while `temperature` and `topP` influence how the model selects output. The application returns the generated text and Bedrock's token-usage counts as JSON.

## 5. Send prompts through the web API

The `ask` helper serialises a prompt as JSON, sends it in an HTTP `POST` request, waits for the response, and decodes the returned JSON. The client knows only the API URL and contract; API Gateway, Lambda, and Bedrock implement the server-side path.

The first prompt asks the model to explain a concept for a specified audience and length. The second uses the same endpoint and model for classification. This flexibility comes from the prompt, not from deploying or training a different model.

In [ ]:
# Later, this timestamp limits the CloudWatch query to requests from this run.
requests_started_at = datetime.now(timezone.utc)

def ask(prompt):
    # The notebook calls our public API; API Gateway then invokes Lambda.
    request = Request(
        API_URL,
        data=json.dumps({"prompt": prompt}).encode("utf-8"),
        headers={"content-type": "application/json"},
        method="POST",
    )
    # The response body arrives as UTF-8 JSON bytes and must be decoded twice.
    with urlopen(request, timeout=45) as response:
        return json.loads(response.read().decode("utf-8"))

def print_answer(answer):
    print(f"Model: {answer['model']}")
    print("Response:")
    print(answer["response"])
    # Token counts are useful for understanding both context size and model cost.
    usage = answer.get("usage", {})
    if usage:
        print("Token usage:")
        print(f"  Input: {usage.get('inputTokens', 'not reported')}")
        print(f"  Output: {usage.get('outputTokens', 'not reported')}")
        print(f"  Total: {usage.get('totalTokens', 'not reported')}")

Model: amazon.nova-micro-v1:0
Response:
Overfitting occurs when a machine learning model learns the training data too well, including its noise and outliers, which can lead to poor performance on new, unseen data because it fails to generalize what it has learned. Essentially, it's like memorizing answers without understanding the underlying concepts.
Token usage:
  Input: 14
  Output: 57
  Total: 71


In [ ]:
ans = ask("Explain overfitting to a first-year computing student in two sentences.")
print_answer(ans)

Model: amazon.nova-micro-v1:0
Response:
Overfitting occurs when a machine learning model learns the training data too well, including its noise and outliers, which leads to poor performance on new, unseen data because it fails to generalize from the training examples. Essentially, it's like memorizing answers without understanding the underlying concepts.
Token usage:
  Input: 14
  Output: 56
  Total: 70


In [ ]:
ans = ask(
    "Classify this review as positive, negative, or neutral. Return only the label: "
    "The interface is clear but the application is slow."
)
print_answer(ans)

Model: amazon.nova-micro-v1:0
Response:
Neutral
Token usage:
  Input: 27
  Output: 2
  Total: 29


## 6. Inspect CloudWatch evidence

CloudWatch Logs provides operational evidence that the API request reached Lambda and that Lambda completed the Bedrock call. The application records the selected model, prompt length, and token usage, but deliberately does not log the prompt or generated response. Avoiding unnecessary content in logs reduces the risk of retaining sensitive user data.

The following cell filters for the application's `bedrock_invocation_complete` records created after this notebook began sending requests. Runtime startup messages are excluded because they do not demonstrate that the model invocation succeeded.

Lambda creates its CloudWatch log group lazily when the function first writes logs; deploying the stack alone does not create it. If the group is not available yet, the cell reports that state and tells you what to run next.

In [14]:
logs = session.client("logs")
log_group = f"/aws/lambda/{FUNCTION}"

try:
    # Search for the structured application log, not routine Lambda platform logs.
    events = logs.filter_log_events(
        logGroupName=log_group,
        filterPattern='"bedrock_invocation_complete"',
        startTime=int(requests_started_at.timestamp() * 1000),
        limit=10,
    )["events"]
except logs.exceptions.ResourceNotFoundException:
    print(f"CloudWatch log group {log_group} does not exist yet.")
    print("Lambda creates the group on its first invocation. Run the prompt cells above, wait a few seconds, then rerun this cell.")
else:
    if not events:
        print("The log group exists, but no completed Bedrock invocation is visible yet. Wait a few seconds, then rerun this cell.")
    else:
        print("Lambda workflow evidence:")
        for event in events:
            payload = event["message"].split("bedrock_invocation_complete ", 1)[-1]
            evidence = json.loads(payload)
            print(f"Model: {evidence['model_id']}")
            print(f"Prompt characters: {evidence['prompt_chars']}")
            usage = evidence.get("usage", {})
            if usage:
                print(f"Tokens: {usage.get('inputTokens', 0)} input, {usage.get('outputTokens', 0)} output, {usage.get('totalTokens', 0)} total")
            print("---")

Lambda workflow evidence:
Model: amazon.nova-micro-v1:0
Prompt characters: 71
Tokens: 14 input, 57 output, 71 total
---
Model: amazon.nova-micro-v1:0
Prompt characters: 130
Tokens: 27 input, 2 output, 29 total
---
Model: amazon.nova-micro-v1:0
Prompt characters: 71
Tokens: 14 input, 56 output, 70 total
---


## What this demo demonstrates

A foundation model is a flexible managed component, but it is not the complete application. We selected the model, designed the prompts and HTTP contract, validated inputs, configured inference parameters, limited permissions and capacity, handled errors, and decided what to log. AWS operates the API Gateway, Lambda, and Bedrock infrastructure and hosts the model.

The same model performed explanation and classification because each prompt supplied a different task. Prompting changes the request and influences the response; it does not update the model weights. Generated output can vary and should be evaluated rather than assumed to be correct.

## 7. Clean up the AWS resources

When you have finished the demo, delete the stack so that the unauthenticated public endpoint does not remain available. The `--yes` flag is required because teardown is destructive.

In [ ]:
profile_option = f" --profile {PROFILE}" if PROFILE else ""
print("When you have finished the demo, run:")
print(f"cd {DEMO_ROOT} && ./manage_demo.sh --demo 2 --action teardown --yes{profile_option}")